In [7]:
import json
import csv
import re
from typing import Dict, List, Any

def load_movie_mapping(csv_file: str) -> Dict[str, str]:
    """Load movie ID to name mapping from CSV file."""
    movie_mapping = {}
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            movie_mapping[row['movieId']] = row['movieName']
    return movie_mapping

def clean_movie_name(movie_name: str) -> str:
    """Remove year and parentheses from movie name."""
    # Remove year patterns like (2003), (1994-1995), etc.
    return re.sub(r'\s*\([^)]*\)\s*$', '', movie_name).strip()

def replace_movie_mentions(text: str, movie_mapping: Dict[str, str]) -> str:
    """Replace @movieId mentions with actual movie names in text, removing years."""
    def replace_match(match):
        movie_id = match.group(1)
        full_movie_name = movie_mapping.get(movie_id, f"@{movie_id}")
        return clean_movie_name(full_movie_name)
    
    return re.sub(r'@(\d+)', replace_match, text)

def extract_ground_truth(data: Dict[str, Any], movie_mapping: Dict[str, str]) -> str:
    """Extract ground truth recommendations from the dataset."""
    ground_truth_movies = []
    
    # Check respondent questions for suggested movies that haven't been seen
    respondent_questions = data.get('respondentQuestions', {})
    if isinstance(respondent_questions, dict):
        for movie_id, info in respondent_questions.items():
            if info.get('suggested', 0) == 1 and info.get('seen', 1) == 0:
                movie_name = movie_mapping.get(movie_id, f"Unknown Movie ({movie_id})")
                # Keep full name with year for ground truth
                if movie_name not in ground_truth_movies:
                    ground_truth_movies.append(movie_name)
    
    # Check initiator questions for suggested movies that haven't been seen
    initiator_questions = data.get('initiatorQuestions', {})
    if isinstance(initiator_questions, dict):
        for movie_id, info in initiator_questions.items():
            if info.get('suggested', 0) == 1 and info.get('seen', 1) == 0:
                movie_name = movie_mapping.get(movie_id, f"Unknown Movie ({movie_id})")
                # Keep full name with year for ground truth
                if movie_name not in ground_truth_movies:
                    ground_truth_movies.append(movie_name)
    
    return ", ".join(ground_truth_movies) if ground_truth_movies else "No recommendations"

def merge_consecutive_turns(conversation: List[Dict[str, str]]) -> List[Dict[str, str]]:
    """Merge consecutive turns from the same role."""
    if not conversation:
        return conversation
    
    merged_conversation = []
    current_turn = conversation[0].copy()
    
    for i in range(1, len(conversation)):
        next_turn = conversation[i]
        
        if current_turn['role'] == next_turn['role']:
            # Merge with current turn
            current_turn['content'] += ' ' + next_turn['content']
        else:
            # Different role, save current turn and start new one
            merged_conversation.append(current_turn)
            current_turn = next_turn.copy()
    
    # Don't forget the last turn
    merged_conversation.append(current_turn)
    
    return merged_conversation

def convert_messages_to_conversation(messages: List[Dict[str, Any]], movie_mapping: Dict[str, str], 
                                    initiator_id: int, respondent_id: int) -> List[Dict[str, str]]:
    """Convert messages to conversation format with role-based structure."""
    conversation = []
    
    for message in messages:
        sender_id = message['senderWorkerId']
        
        # Map worker IDs to roles
        if sender_id == initiator_id:
            role = 'user'
        elif sender_id == respondent_id:
            role = 'assistant'
        else:
            role = 'unknown'
            print(f"WARNING: Unexpected senderWorkerId {sender_id}")
        
        content = replace_movie_mentions(message['text'], movie_mapping)
        conversation.append({'role': role, 'content': content})
    
    # Merge consecutive turns from the same role
    return merge_consecutive_turns(conversation)

def process_redial_dataset(jsonl_file: str, movies_csv: str, output_csv: str):
    """Process the ReDial dataset and convert to target CSV format."""
    
    # Load movie mapping
    print("Loading movie mapping...")
    movie_mapping = load_movie_mapping(movies_csv)
    print(f"Loaded {len(movie_mapping)} movie mappings")
    
    # Process conversations
    print("Processing conversations...")
    processed_conversations = []
    skipped = 0
    
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if not line.strip():
                continue
                
            try:
                data = json.loads(line)
                
                # Extract and validate worker IDs
                conversation_id = data.get('conversationId', '')
                initiator_id = data.get('initiatorWorkerId')
                respondent_id = data.get('respondentWorkerId')
                
                if initiator_id is None or respondent_id is None or initiator_id == respondent_id:
                    print(f"WARNING: Invalid worker IDs in conversation {conversation_id}")
                    skipped += 1
                    continue
                
                # Process conversation
                conversation = convert_messages_to_conversation(
                    data['messages'], movie_mapping, initiator_id, respondent_id
                )
                ground_truth = extract_ground_truth(data, movie_mapping)
                
                # Skip conversations with no recommendations
                if ground_truth == "No recommendations":
                    skipped += 1
                    continue
                
                processed_conversations.append({
                    'dialog_id': conversation_id,
                    'ground_truth': ground_truth,
                    'conversation': str(conversation)
                })
                
                if line_num % 100 == 0:
                    print(f"Processed {line_num} conversations...")
                    
            except json.JSONDecodeError as e:
                print(f"Error parsing line {line_num}: {e}")
                skipped += 1
                continue
    
    # Write to CSV
    print(f"Writing {len(processed_conversations)} conversations to CSV...")
    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['dialog_id', 'ground_truth', 'conversation'], 
                               quoting=csv.QUOTE_ALL)
        writer.writeheader()
        writer.writerows(processed_conversations)
    
    print(f"Complete! Processed: {len(processed_conversations)}, Skipped: {skipped}")

In [9]:

jsonl_file = "raw/test_data.jsonl"
movies_csv = "raw/movies_with_mentions.csv"
output_csv = "multiturn_form/test.csv"

try:
    process_redial_dataset(jsonl_file, movies_csv, output_csv)
except FileNotFoundError as e:
    print(f"File not found: {e}")
except Exception as e:
    print(f"Error: {e}")


Loading movie mapping...
Loaded 6924 movie mappings
Processing conversations...
Processed 100 conversations...
Processed 200 conversations...
Processed 300 conversations...
Processed 400 conversations...
Processed 500 conversations...
Processed 600 conversations...
Processed 800 conversations...
Processed 900 conversations...
Processed 1200 conversations...
Processed 1300 conversations...
Writing 1076 conversations to CSV...
Complete! Processed: 1076, Skipped: 266
